## Determining the optimal number of hidden layera and neuron for an ANN

- start simple: begin with a simple architecture and gradually increase complexity if needed
- grid search/random search: use either to try different archs
- cross-validation: use it to evaluate performance of different archs
- heuristics and rule of thumb:
    - number of neurons in hidden layers should be b/w size of input layer and output layer
    - start with 1-2 hidden layers

In [2]:
import pandas as pd
import tensorflow as tf
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
data = pd.read_csv("Churn_Modelling.csv")
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data=pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [4]:
# define a function to create a model and try different parameters using Keras Classifier

def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1], )))

    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))
    
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [10]:
# create a keras classifier
model = KerasClassifier(layers=1, neurons=32, build_fn=create_model, verbose=1)

In [11]:
# grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [12]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3, verbose=1)
grid_result = grid.fit(X_train, y_train)

print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/y0a02i0/Documents/Udemy/Complete Gen AI

Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7790 - loss: 0.5452
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7729 - loss: 0.4938
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7821 - loss: 0.4960
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7920 - loss: 0.4838
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7456 - loss: 0.5289
  1/167 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.6875 - loss: 0.5926Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7274 - loss: 0.5516
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6409 - loss: 0.6447
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7395 - loss: 0.5406  
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8035 - loss: 0.4592
 86/167 ━━━━━━━━━━━━━━━━━━━

/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/stepep - accuracy: 0.8946 - loss: 0.2917 
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/stepep - accuracy: 0.8707 - loss: 0.3135
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/stepep - accuracy: 0.8876 - loss: 0.29
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8761 - loss: 0.3034
Epoch 1/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/stepep - accuracy: 0.8766 - loss: 0.3027
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


...

167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8832 - loss: 0.2884
167/167 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8046 - loss: 0.4548
Epoch 2/100
 85/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7847 - loss: 0.5339Epoch 1/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8397 - loss: 0.3869
Epoch 5/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8264 - loss: 0.4095
Epoch 5/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8318 - loss: 0.4008
Epoch 7/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8061 - loss: 0.4616
Epoch 2/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8116 - loss: 0.4270
  1/167 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9062 - loss: 0.3290Epoch 3/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8268 - loss: 0.4117
Epoch 3/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8534 - loss: 0.3669 
Epoch 7/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/stepep - accuracy: 0.8324 - loss:

/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8315 - loss: 0.3987
Epoch 6/100
 21/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8253 - loss: 0.4182 Epoch 1/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8479 - loss: 0.3690 
Epoch 6/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8361 - loss: 0.3882
Epoch 8/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8483 - loss: 0.3808
Epoch 4/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8339 - loss: 0.4042 
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8267 - loss: 0.4020
Epoch 4/100
 37/167 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8682 - loss: 0.3609 Epoch 3/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8575 - loss: 0.35627
Epoch 4/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8344 - loss: 0.3910
Epoch 3/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8554 - loss: 0.3567
101/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8383 - lo

/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/y0a02i0/Documents/Udemy/Complete Gen AI Course with Langchain and HuggingFace/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 440us/step - accuracy: 0.6183 - loss: 0.6635
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 459us/step - accuracy: 0.8050 - loss: 0.4609
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 471us/step - accuracy: 0.8158 - loss: 0.4283
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 456us/step - accuracy: 0.8232 - loss: 0.4118
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 448us/step - accuracy: 0.8301 - loss: 0.3995
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 451us/step - accuracy: 0.8341 - loss: 0.3889
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 449us/step - accuracy: 0.8378 - loss: 0.3803
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 485us/step - accuracy: 0.8414 - loss: 0.3735
Epoch 9/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 433us/step - accuracy: 0.8430 - loss: 0.3680
Epoch 10/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 443us/step - accuracy: 0.8446 - loss: 0.3638
Epoch 11/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 478us/step - accuracy: 0.8462 - loss: 0.3610
Epoch 12/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 